# 1. Exploratory Data Analysis: Video Game Sales
**Objective:** Load the raw data, inspect its shape, check the data types, and view the initial rows.

In [37]:
import pandas as pd

# Load the raw dataset
raw_sales_data = pd.read_csv('vgsales.csv')

# 1. Display the shape (number of rows and columns)
print(f"Dataset Shape: {raw_sales_data.shape}\n")

# 2. Display the data types of each column
print("Data Types:")
print(raw_sales_data.dtypes)
print("\n")

# 3. Display the first 5 rows
raw_sales_data.head()

Dataset Shape: (16598, 11)

Data Types:
Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object




,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


## 2. Data Cleaning Pipeline
**Objective:** Hunt down missing values, handle them using an explicit strategy, remove any duplicate records, and create a pristine working copy of the dataset.

In [38]:
# Check for missing values (NaNs) in the dataset
print("--- Missing Values ---")
missing_data = raw_sales_data.isnull().sum()
print(missing_data[missing_data > 0]) # only show those columns which have missing data

# Check for identical duplicate rows
duplicate_count = raw_sales_data.duplicated().sum()
print(f"\nTotal duplicate rows: {duplicate_count}")

--- Missing Values ---
Year         271
Publisher     58
dtype: int64

Total duplicate rows: 0


In [39]:
# Create a fresh copy to avoid overwriting our raw data
clean_sales_data = raw_sales_data.copy()

# STRATEGY FOR MISSING VALUES:
# 1. 'Year': Since predicting a specific release year is highly inaccurate, we will drop rows where 'Year' is missing.
# 2. 'Publisher': We don't want to lose the valid sales data for these rows, so we will fill missing publishers with 'Unknown'.

# Apply the strategy
clean_sales_data = clean_sales_data.dropna(subset=['Year'])  # Drop rows where 'Year' is missing
clean_sales_data['Publisher'] = clean_sales_data['Publisher'].fillna('Unknown') # Fill missing 'Publisher' with 'Unknown'

# We will drop all duplicate rows to ensure the dataset is clean and each entry is unique.
# In this dataset, there are no duplicate rows, but this step is included for completeness.
clean_sales_data = clean_sales_data.drop_duplicates()

# Final Verification
print(f"New Dataset Shape: {clean_sales_data.shape}")
print(f"Remaining Missing Values: {clean_sales_data.isnull().sum().max()}") # will give 0 if cleaning went well

New Dataset Shape: (16327, 11)
Remaining Missing Values: 0


## 3. Data Aggregation & Business Insights
**Objective:** Group the data to discover which game genres and platforms generate the most global revenue.

In [40]:
# Group by Genre and calculate total global sales
genre_sales = clean_sales_data.groupby('Genre')['Global_Sales'].sum().reset_index()

# Sort the values to see the highest-grossing genres first
top_genres = genre_sales.sort_values(by='Global_Sales', ascending=False)

print("--- All Video Game Genres by Global Sales (Millions) ---")
top_genres

--- All Video Game Genres by Global Sales (Millions) ---


,Genre,Global_Sales
0,Action,1722.88
10,Sports,1309.24
8,Shooter,1026.20
7,Role-Playing,923.84
4,Platform,829.15
3,Misc,797.62
6,Racing,726.77
2,Fighting,444.05
9,Simulation,390.16
5,Puzzle,242.22


### Regional Revenue by Top Publishers
**Business Question:** Which game publishers dominate the global market, and is their total revenue driven more by North American or Japanese sales?

**Objective:** Group the data by Publisher and aggregate multiple regional sales columns to compare market dependencies.

In [41]:
# Group by Publisher and sum up sales for multiple specific regions
publisher_sales = clean_sales_data.groupby('Publisher')[['NA_Sales', 'JP_Sales','Global_Sales']].sum().reset_index()

# Sort the values to see the highest-grossing publishers first
top_publishers = publisher_sales.sort_values(by='Global_Sales', ascending=False)

print("--- Top 10 Game Publishers: Regional Breakdown (Millions) ---")
top_publishers.head(10)

--- Top 10 Game Publishers: Regional Breakdown (Millions) ---


,Publisher,NA_Sales,JP_Sales,Global_Sales
359,Nintendo,815.75,454.99,1784.43
138,Electronic Arts,584.22,13.98,1093.39
21,Activision,426.01,6.54,721.41
455,Sony Computer Entertainment,265.22,74.10,607.28
524,Ubisoft,252.81,7.33,473.54
493,Take-Two Interactive,220.47,5.83,399.30
487,THQ,208.60,5.01,340.44
275,Konami Digital Entertainment,88.91,90.93,278.56
445,Sega,108.78,56.19,270.70
347,Namco Bandai Games,69.38,126.84,253.65


## 4. Advanced Transformation: Pivot Tables
**Business Question:** How do different game genres perform when broken down by specific global regions?

**Objective:** Create a pivot table to compare North American, European, and Japanese sales across all genres simultaneously.

In [42]:
# Build a pivot table to show the total sales for each genre in North America, Japan, and Europe
regional_genre_pivot = pd.pivot_table(
    data = clean_sales_data,
    values = ['NA_Sales','JP_Sales','EU_Sales'],
    index = 'Genre',
    aggfunc = 'sum'
)
# Sort it by North American sales to make it easy to read
regional_genre_pivot = regional_genre_pivot.sort_values(by='NA_Sales', ascending=False)

print("--- Regional Sales Breakdown by Genre (Millions) ---")
regional_genre_pivot


--- Regional Sales Breakdown by Genre (Millions) ---


,EU_Sales,JP_Sales,NA_Sales
Genre,,,
Action,516.48,158.66,861.80
Sports,371.34,134.76,670.09
Shooter,310.45,38.18,575.16
Platform,200.67,130.65,445.99
Misc,213.82,106.67,402.48
Racing,236.32,56.61,356.93
Role-Playing,187.58,350.29,326.50
Fighting,100.00,87.15,220.74
Simulation,113.20,63.54,181.78


### Advanced Pivot: Genre Trends Over Time
**Business Question:** How has the popularity (measured by Global Sales) of different game genres evolved over the years?
**Objective:** Create a 2D pivot table with Genres as rows and Release Years as columns to map out industry trends over time.

In [ ]:
# Build a pivot table to show the total sales for each genre by year
genre_year_pivot = pd.pivot_table(
    data = clean_sales_data,
    values = 'Global_Sales',
    index = 'Genre',
    columns = 'Year',
    aggfunc = 'sum'
)

genre_year_pivot = genre_year_pivot.fillna(0) # Fill NaN values with 0 for better visualization

# Grab all rows, but only colums from 2000 to 2015 
# (Since this dataset has data from 1980 to 2016, we will focus on the more recent years for better insights)

recent_years_pivot = genre_year_pivot.loc[:, 2000.0:2015.0] # from 2000 to 2015 
recent_years_pivot


Year,2000.0,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,2010.0,2011.0,2012.0,2013.0,2014.0,2015.0
Genre,,,,,,,,,,,,,,,,
Action,34.04,59.39,86.77,67.93,76.26,85.69,66.58,106.50,136.39,139.36,117.64,118.96,122.04,125.22,99.02,70.70
Adventure,2.98,9.12,11.05,2.14,8.70,8.53,11.47,24.47,25.02,20.68,16.57,15.98,5.99,6.61,6.06,8.03
Fighting,20.22,18.12,25.02,23.73,16.78,19.72,22.55,17.61,35.38,32.15,14.89,22.68,9.51,7.21,16.15,7.78
Misc,15.54,16.40,15.67,23.82,31.32,61.24,67.35,92.27,87.03,76.94,96.86,56.08,22.92,25.65,23.68,11.69
Platform,16.06,39.28,45.97,42.89,47.34,23.56,49.80,35.59,35.70,41.09,31.90,28.11,18.55,25.12,8.89,6.05
Puzzle,3.82,8.00,5.34,2.42,8.43,20.45,10.90,24.00,15.59,20.31,11.18,5.11,1.76,0.99,1.50,0.70
Racing,19.99,55.81,30.20,52.19,47.86,56.42,34.09,39.17,70.66,34.19,34.93,35.01,14.46,13.04,16.69,7.92
Role-Playing,29.03,22.06,45.13,30.28,53.95,28.55,57.73,43.89,59.83,47.90,70.52,53.37,47.81,44.92,45.86,36.44
Shooter,6.81,24.77,48.58,27.14,46.95,41.60,38.37,71.04,59.51,69.89,77.41,99.36,72.86,62.80,66.00,66.15
